In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="gbkaUuIbTyDEFUiAXskS")
project = rf.workspace("kasha").project("detect-mcgji")
version = project.version(1)
dataset = version.download("yolov8")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 68.3 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.15
    Uninstalling idna-3.15:
      Successfully uninstalled idna-3.15
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to detect-1 in yolov8:: 100%|██████████| 423/423 [00:00<00:00, 6098.64it/s]


## Step 1: Install Libraries and Download COCO Images

First, we'll ensure all necessary libraries are installed, including `ultralytics` for YOLOv8. Then, we'll download the COCO 2017 training and validation images, as these were the missing components in your previous pipeline. We'll extract them to a dedicated directory for raw COCO images.

In [3]:
import os
import shutil
import json
import yaml
import random
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

# Define paths
RAW_COCO_IMAGE_DIR = "/content/coco_raw_images"
COCO_SUBSET_YOLO_DIR = "/content/coco_subset_yolo" # This will be the processed COCO dataset in YOLO format
CUSTOM_DIR = "/content/detect-1"
MERGED_DIR = "/content/merged_dataset"

# Create directory for raw COCO images
os.makedirs(RAW_COCO_IMAGE_DIR, exist_ok=True)

print("Downloading COCO 2017 Train Images...")
!wget --no-check-certificate http://images.cocodataset.org/zips/train2017.zip -O {RAW_COCO_IMAGE_DIR}/train2017.zip
!unzip -q {RAW_COCO_IMAGE_DIR}/train2017.zip -d {RAW_COCO_IMAGE_DIR}

print("Downloading COCO 2017 Val Images...")
!wget --no-check-certificate http://images.cocodataset.org/zips/val2017.zip -O {RAW_COCO_IMAGE_DIR}/val2017.zip
!unzip -q {RAW_COCO_IMAGE_DIR}/val2017.zip -d {RAW_COCO_IMAGE_DIR}

print("COCO images downloaded and extracted.")

print("Downloading COCO 2017 Annotations...")
!wget --no-check-certificate http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O /content/annotations_trainval2017.zip
!unzip -q /content/annotations_trainval2017.zip -d /content/

# Load COCO annotations for both train and val
with open("annotations/instances_train2017.json", "r") as f:
    coco_train_ann = json.load(f)

with open("annotations/instances_val2017.json", "r") as f:
    coco_val_ann = json.load(f)

print("COCO annotations loaded.")

# Ensure Roboflow dataset directory exists (it should be from previous steps)
os.makedirs(CUSTOM_DIR, exist_ok=True)

print("Setup and initial downloads complete.")

--2026-06-01 18:32:26--  http://images.cocodataset.org/zips/train2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 16.15.207.28, 16.182.97.217, 16.15.228.75, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|16.15.207.28|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 19336861798 (18G) [application/zip]
Saving to: ‘/content/coco_raw_images/train2017.zip’

/content/coco_raw_i 100%[===================>]  18.01G  40.1MB/s    in 7m 13s  

2026-06-01 18:39:40 (42.6 MB/s) - ‘/content/coco_raw_images/train2017.zip’ saved [19336861798/19336861798]

replace /content/coco_raw_images/train2017/000000147328.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
--2026-06-01 18:43:20--  http://images.cocodataset.org/zips/val2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 16.15.253.37, 16.15.207.180, 16.15.212.255, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|16.15.253.37|:80... connected.
HTTP request sent

## Step 2: Select COCO Subset and Convert to YOLOv8 Format

This step is crucial for creating a balanced COCO subset that meets your criteria: approximately 6000 images, all COCO classes represented, and at least 80 images per class. We will develop a script to:

1.  **Combine Annotations**: Merge `train2017` and `val2017` annotations for unified processing.
2.  **Filter Images**: Select images based on the specified class representation and quantity.
3.  **Convert to YOLOv8 Format**: Transform COCO's `[x, y, width, height]` bounding box format to YOLO's `[class_id, x_center, y_center, width_normalized, height_normalized]` format.
4.  **Organize Files**: Create a new directory structure (`/content/coco_subset_yolo`) with the selected images and their corresponding YOLO label files, ready for merging.

In [4]:
# Constants for COCO subset selection
NUM_COCO_IMAGES_TARGET = 6000
MIN_IMAGES_PER_CLASS = 80
VAL_SPLIT_RATIO = 0.2 # 20% for validation in the COCO subset

# --- Helper Functions ---

def get_image_info_from_annotations(annotations_dict):
    """Extracts image metadata and category IDs from COCO annotations."""
    img_id_to_info = {img['id']: {'file_name': img['file_name'], 'width': img['width'], 'height': img['height'], 'categories': set()} for img in annotations_dict['images']}
    img_id_to_anns = defaultdict(list)
    for ann in annotations_dict['annotations']:
        img_id_to_anns[ann['image_id']].append(ann)
        if ann['image_id'] in img_id_to_info:
            img_id_to_info[ann['image_id']]['categories'].add(ann['category_id'])

    # Filter out images without annotations or missing info
    filtered_image_info = {}
    for img_id, info in img_id_to_info.items():
        if info['categories'] and img_id_to_anns[img_id]:
            filtered_image_info[img_id] = info
            filtered_image_info[img_id]['annotations'] = img_id_to_anns[img_id]

    return filtered_image_info, {cat['id']: cat['name'] for cat in annotations_dict['categories']}

def convert_coco_to_yolo(annotations, image_width, image_height, output_filepath):
    """Converts COCO annotations for a single image to YOLO format and saves to file."""
    yolo_lines = []
    for ann in annotations:
        # COCO format: [x_min, y_min, width, height]
        x_min, y_min, bbox_width, bbox_height = ann['bbox']

        # Calculate YOLO format: [x_center, y_center, width_normalized, height_normalized]
        x_center = (x_min + bbox_width / 2) / image_width
        y_center = (y_min + bbox_height / 2) / image_height
        width_normalized = bbox_width / image_width
        height_normalized = bbox_height / image_height

        # Ensure values are within [0, 1] range
        x_center = max(0.0, min(1.0, x_center))
        y_center = max(0.0, min(1.0, y_center))
        width_normalized = max(0.0, min(1.0, width_normalized))
        height_normalized = max(0.0, min(1.0, height_normalized))

        # COCO category ID might not be contiguous, map to 0-indexed new ID later if needed.
        # For now, use original category_id as placeholder. We'll remap during merge.
        class_id = ann['category_id'] # This will be remapped during the merge step

        yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width_normalized:.6f} {height_normalized:.6f}")

    if yolo_lines:
        with open(output_filepath, "w") as f:
            f.write("\n".join(yolo_lines))
    else:
        # If no annotations, create an empty label file (or skip, depending on preference)
        open(output_filepath, "a").close()

# --- Main Subset Selection and Conversion Logic ---

print("Processing COCO annotations for subset selection...")

# Combine image info and categories from both train and val
all_image_info = {}
all_coco_categories = {}

train_img_info, train_cats = get_image_info_from_annotations(coco_train_ann)
val_img_info, val_cats = get_image_info_from_annotations(coco_val_ann)

all_image_info.update(train_img_info)
all_image_info.update(val_img_info)
all_coco_categories.update(train_cats)
all_coco_categories.update(val_cats)

# Reverse map COCO original cat IDs to contiguous 0-indexed IDs for this subset
# This temporary mapping is important for our selection logic and will be remapped again during merging
coco_original_cat_ids = sorted(all_coco_categories.keys())
coco_temp_cat_id_map = {old_id: new_id for new_id, old_id in enumerate(coco_original_cat_ids)}
coco_temp_names = {new_id: all_coco_categories[old_id] for old_id, new_id in coco_temp_cat_id_map.items()}

# Group images by categories they contain
cat_to_images = defaultdict(list)
for img_id, info in all_image_info.items():
    for original_cat_id in info['categories']:
        cat_to_images[coco_temp_cat_id_map[original_cat_id]].append(img_id)

selected_image_ids = set()
images_per_cat_count = defaultdict(int)

# 1. Ensure minimum images per class
print("Ensuring minimum images per class...")
for temp_cat_id in sorted(cat_to_images.keys()):
    random.shuffle(cat_to_images[temp_cat_id]) # Shuffle to get varied images
    for img_id in cat_to_images[temp_cat_id]:
        if images_per_cat_count[temp_cat_id] < MIN_IMAGES_PER_CLASS:
            selected_image_ids.add(img_id)
            for c_id in all_image_info[img_id]['categories']:
                 images_per_cat_count[coco_temp_cat_id_map[c_id]] += 1
        else:
            break

# Re-count after initial selection to prevent overcounting if one image has multiple categories
images_per_cat_count = defaultdict(int)
for img_id in selected_image_ids:
    for original_cat_id in all_image_info[img_id]['categories']:
        images_per_cat_count[coco_temp_cat_id_map[original_cat_id]] += 1

print(f"Selected {len(selected_image_ids)} images for minimum class representation.")

# 2. Add random images until target count is reached
remaining_image_ids = list(set(all_image_info.keys()) - selected_image_ids)
random.shuffle(remaining_image_ids)

for img_id in tqdm(remaining_image_ids, desc="Adding remaining images", total=len(remaining_image_ids)):
    if len(selected_image_ids) < NUM_COCO_IMAGES_TARGET:
        selected_image_ids.add(img_id)
    else:
        break

print(f"Final COCO subset size: {len(selected_image_ids)} images.")

# --- Create YOLOv8 COCO Subset Directory Structure, Copy Files, and Generate data.yaml ---

print(f"Creating YOLOv8 dataset structure in {COCO_SUBSET_YOLO_DIR}...")
if os.path.exists(COCO_SUBSET_YOLO_DIR):
    shutil.rmtree(COCO_SUBSET_YOLO_DIR)
os.makedirs(COCO_SUBSET_YOLO_DIR, exist_ok=True)
os.makedirs(os.path.join(COCO_SUBSET_YOLO_DIR, "images", "train"), exist_ok=True)
os.makedirs(os.path.join(COCO_SUBSET_YOLO_DIR, "labels", "train"), exist_ok=True)
os.makedirs(os.path.join(COCO_SUBSET_YOLO_DIR, "images", "val"), exist_ok=True)
os.makedirs(os.path.join(COCO_SUBSET_YOLO_DIR, "labels", "val"), exist_ok=True)

# Split selected_image_ids into train and val for the new subset
selected_train_ids = []
selected_val_ids = []

for img_id in selected_image_ids:
    if img_id in train_img_info: # Check if original image was from COCO train set
        selected_train_ids.append(img_id)
    elif img_id in val_img_info: # Check if original image was from COCO val set
        selected_val_ids.append(img_id)
    else:
        print(f"Warning: Image ID {img_id} not found in original train or val info. Skipping.")

# If we didn't get enough validation images, randomly move some from train to val
# This handles cases where MIN_IMAGES_PER_CLASS heavily skews the initial split.
target_val_count = int(len(selected_image_ids) * VAL_SPLIT_RATIO)
if len(selected_val_ids) < target_val_count:
    num_to_move = target_val_count - len(selected_val_ids)
    if num_to_move > len(selected_train_ids):
        num_to_move = len(selected_train_ids) # Don't move more than available

    random.shuffle(selected_train_ids)
    moved_ids = selected_train_ids[:num_to_move]
    selected_val_ids.extend(moved_ids)
    selected_train_ids = selected_train_ids[num_to_move:]

print(f"Distributing {len(selected_train_ids)} images to train and {len(selected_val_ids)} to val for COCO subset.")

# Process and copy files for the train split
print("Processing COCO subset train split...")
for img_id in tqdm(selected_train_ids, desc="Processing COCO Train"):
    img_info = all_image_info[img_id]
    original_file_name = img_info['file_name']

    # Determine original split (train2017 or val2017) to construct correct source path
    if img_id in train_img_info:
        original_coco_split_folder = "train2017"
    elif img_id in val_img_info:
        original_coco_split_folder = "val2017"
    else:
        # This case should ideally not happen if selected_image_ids only contains valid IDs
        print(f"Error: Image ID {img_id} not found in original train_img_info or val_img_info. Skipping image {original_file_name}.")
        continue

    src_img_path = os.path.join(RAW_COCO_IMAGE_DIR, original_coco_split_folder, Path(original_file_name).name)
    dst_img_path = os.path.join(COCO_SUBSET_YOLO_DIR, "images", "train", Path(original_file_name).name)
    dst_lbl_path = os.path.join(COCO_SUBSET_YOLO_DIR, "labels", "train", Path(original_file_name).stem + ".txt")

    if not os.path.exists(src_img_path):
        print(f"Warning: Source image {src_img_path} not found. Skipping.")
        continue

    shutil.copy2(src_img_path, dst_img_path)
    convert_coco_to_yolo(
        img_info['annotations'],
        img_info['width'],
        img_info['height'],
        dst_lbl_path
    )

# Process and copy files for the val split
print("Processing COCO subset val split...")
for img_id in tqdm(selected_val_ids, desc="Processing COCO Val"):
    img_info = all_image_info[img_id]
    original_file_name = img_info['file_name']

    # Determine original split (train2017 or val2017) to construct correct source path
    if img_id in train_img_info:
        original_coco_split_folder = "train2017"
    elif img_id in val_img_info:
        original_coco_split_folder = "val2017"
    else:
        # This case should ideally not happen if selected_image_ids only contains valid IDs
        print(f"Error: Image ID {img_id} not found in original train_img_info or val_img_info. Skipping image {original_file_name}.")
        continue

    src_img_path = os.path.join(RAW_COCO_IMAGE_DIR, original_coco_split_folder, Path(original_file_name).name)
    dst_img_path = os.path.join(COCO_SUBSET_YOLO_DIR, "images", "val", Path(original_file_name).name)
    dst_lbl_path = os.path.join(COCO_SUBSET_YOLO_DIR, "labels", "val", Path(original_file_name).stem + ".txt")

    if not os.path.exists(src_img_path):
        print(f"Warning: Source image {src_img_path} not found. Skipping.")
        continue

    shutil.copy2(src_img_path, dst_img_path)
    convert_coco_to_yolo(
        img_info['annotations'],
        img_info['width'],
        img_info['height'],
        dst_lbl_path
    )

print("COCO subset image and label files generated.")

# Generate data.yaml for the COCO subset
coco_subset_yaml = {
    "path": COCO_SUBSET_YOLO_DIR,
    "train": "images/train",
    "val": "images/val",
    "nc": len(coco_temp_names),
    "names": [coco_temp_names[i] for i in sorted(coco_temp_names.keys())] # Ensure names are ordered by their temp_cat_id
}

with open(os.path.join(COCO_SUBSET_YOLO_DIR, "data.yaml"), "w") as f:
    yaml.dump(coco_subset_yaml, f, sort_keys=False)

print(f"Generated {COCO_SUBSET_YOLO_DIR}/data.yaml")
print("-" * 50)
print("COCO subset selection and conversion complete.")

Processing COCO annotations for subset selection...
Ensuring minimum images per class...
Selected 3566 images for minimum class representation.


Adding remaining images:   2%|▏         | 2434/118652 [00:00<00:00, 367452.61it/s]


Final COCO subset size: 6000 images.
Creating YOLOv8 dataset structure in /content/coco_subset_yolo...
Distributing 4800 images to train and 1200 to val for COCO subset.
Processing COCO subset train split...


Processing COCO Train: 100%|██████████| 4800/4800 [00:30<00:00, 158.80it/s]


Processing COCO subset val split...


Processing COCO Val: 100%|██████████| 1200/1200 [00:06<00:00, 177.12it/s]

COCO subset image and label files generated.
Generated /content/coco_subset_yolo/data.yaml
--------------------------------------------------
COCO subset selection and conversion complete.


## Step 3: Merge COCO Subset with Roboflow Dataset

Now that we have a properly formatted COCO subset, we can proceed with merging it with your Roboflow dataset. This script will:

1.  **Load `data.yaml` files**: Read class names from both the COCO subset and your custom Roboflow dataset.
2.  **Build a Unified Class List**: Create a comprehensive list of all unique class names, ensuring no duplicates and handling overlapping names correctly.
3.  **Remap Class IDs**: Assign new, contiguous class IDs to all images and labels based on the unified class list.
4.  **Copy and Remap Files**: Copy images and update label files (with the new class IDs) from both source datasets into the new `merged_dataset` directory structure.
5.  **Generate Merged `data.yaml`**: Create a final `data.yaml` file for the combined dataset, ready for YOLOv8 training.

In [5]:
# =====================================================
# PATHS (re-defining for clarity, should match previous cells)
# =====================================================
COCO_DIR_FOR_MERGE = COCO_SUBSET_YOLO_DIR # Use the newly created COCO YOLO subset
CUSTOM_DIR = "/content/detect-1"
MERGED_DIR = "/content/merged_dataset"

# Ensure MERGED_DIR is clean before starting the merge process
if os.path.exists(MERGED_DIR):
    shutil.rmtree(MERGED_DIR)
os.makedirs(MERGED_DIR, exist_ok=True)

# =====================================================
# LOAD CLASS NAMES
# =====================================================
print("Loading data.yaml files...")

with open(os.path.join(COCO_DIR_FOR_MERGE, "data.yaml"), "r") as f:
    coco_yaml = yaml.safe_load(f)

with open(os.path.join(CUSTOM_DIR, "data.yaml"), "r") as f:
    custom_yaml = yaml.safe_load(f)

coco_names = coco_yaml["names"]
custom_names = custom_yaml["names"]

# =====================================================
# BUILD UNIFIED CLASS LIST
# =====================================================
merged_names = list(coco_names)

for cls_name in custom_names:
    if cls_name not in merged_names:
        merged_names.append(cls_name)

merged_class_to_id = {
    cls_name: idx
    for idx, cls_name in enumerate(merged_names)
}

print(f"Unified class list created with {len(merged_names)} classes: {merged_names}")

# =====================================================
# MAP OLD IDS -> NEW IDS
# =====================================================
coco_id_map = {
    original_coco_cat_id: merged_class_to_id[name]
    for original_coco_cat_id, name in all_coco_categories.items() # Use all_coco_categories from previous cell
}

custom_id_map = {
    old_id: merged_class_to_id[name]
    for old_id, name in enumerate(custom_names)
}

# =====================================================
# CREATE OUTPUT FOLDERS
# =====================================================
for split in ["train", "val"]:
    os.makedirs(os.path.join(MERGED_DIR, "images", split), exist_ok=True)
    os.makedirs(os.path.join(MERGED_DIR, "labels", split), exist_ok=True)

# =====================================================
# COPY + REMAP FUNCTION
# =====================================================
def process_split_files(
    src_img_base_dir, # e.g., /content/coco_subset_yolo/images/train
    src_lbl_base_dir, # e.g., /content/coco_subset_yolo/labels/train
    id_map,
    prefix, # 'coco' or 'custom'
    split_name # 'train' or 'val'
):
    print(f"Processing {prefix} {split_name} split from {src_img_base_dir}...")

    if not os.path.exists(src_img_base_dir) or not os.listdir(src_img_base_dir):
        print(f"Skipping {split_name} split for {prefix}: no images found in {src_img_base_dir}")
        return

    # Destination directories are already created at MERGED_DIR/images/{split} and MERGED_DIR/labels/{split}
    merged_img_dir = os.path.join(MERGED_DIR, "images", split_name)
    merged_lbl_dir = os.path.join(MERGED_DIR, "labels", split_name)

    for img_file in tqdm(os.listdir(src_img_base_dir), desc=f"Copying and remapping {prefix} {split_name} files"):
        src_img_path = os.path.join(src_img_base_dir, img_file)

        # Generate unique name to avoid clashes
        new_img_name = f"{prefix}_{img_file}"
        dst_img_path = os.path.join(merged_img_dir, new_img_name)
        shutil.copy2(src_img_path, dst_img_path)

        stem = Path(img_file).stem
        src_lbl_path = os.path.join(src_lbl_base_dir, stem + ".txt")
        dst_lbl_path = os.path.join(merged_lbl_dir, Path(new_img_name).stem + ".txt")

        if not os.path.exists(src_lbl_path):
            # If no label file, create an empty one to match image (or handle as error)
            open(dst_lbl_path, 'a').close()
            continue

        new_lines = []
        with open(src_lbl_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue # Skip malformed lines

                old_class_id = int(parts[0])

                # Remap class ID
                if old_class_id in id_map:
                    new_class_id = id_map[old_class_id]
                    parts[0] = str(new_class_id)
                    new_lines.append(" ".join(parts))
                else:
                    print(f"Warning: Class ID {old_class_id} not found in map for {src_lbl_path}")

        with open(dst_lbl_path, "w") as f:
            f.write("\n".join(new_lines))

# =====================================================
# PROCESS BOTH DATASETS
# =====================================================
print("\nStarting dataset merging...")

for split in ["train", "val"]:
    # COCO Subset Paths
    coco_img_src_dir = os.path.join(COCO_DIR_FOR_MERGE, coco_yaml[split])
    coco_lbl_src_dir = os.path.join(COCO_DIR_FOR_MERGE, coco_yaml[split].replace('images', 'labels'))

    process_split_files(coco_img_src_dir, coco_lbl_src_dir, coco_id_map, "coco", split)

    # Custom Roboflow Dataset Paths
    # The custom_yaml['train'] is '../train/images' and custom_yaml['val'] is '../valid/images'
    # We need to correctly extract 'train' or 'valid' from these paths and join with CUSTOM_DIR.
    # Example: Path('../train/images').parts will be ('..', 'train', 'images')
    relative_path_parts = Path(custom_yaml[split]).parts
    split_folder_name = relative_path_parts[1] # 'train' or 'valid'

    custom_img_src_dir = os.path.join(CUSTOM_DIR, split_folder_name, "images")
    custom_lbl_src_dir = os.path.join(CUSTOM_DIR, split_folder_name, "labels")

    process_split_files(custom_img_src_dir, custom_lbl_src_dir, custom_id_map, "custom", split)

# =====================================================
# CREATE DATA.YAML FOR MERGED DATASET
# =====================================================
merged_yaml = {
    "path": MERGED_DIR,
    "train": "images/train",
    "val": "images/val",
    "nc": len(merged_names),
    "names": merged_names
}

with open(os.path.join(MERGED_DIR, "data.yaml"), "w") as f:
    yaml.dump(merged_yaml, f, sort_keys=False)

print("=" * 50)
print("MERGE COMPLETE")
print(f"Total classes in merged dataset: {len(merged_names)}")
print(f"Merged dataset located at: {MERGED_DIR}")
print("=" * 50)


Loading data.yaml files...
Unified class list created with 85 classes: ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush', 'PC', 'desk', 'fan', 'floor', 'wall']

Starting dataset merging...


Copying and remapping coco train files: 100%|██████████| 4800/4800 [00:11<00:00, 423.82it/s]


Processing custom train split from /content/detect-1/train/images...


Copying and remapping custom train files: 100%|██████████| 210/210 [00:00<00:00, 289.56it/s]


Processing coco val split from /content/coco_subset_yolo/images/val...


Copying and remapping coco val files: 100%|██████████| 1200/1200 [00:05<00:00, 226.58it/s]

Processing custom val split from /content/detect-1/valid/images...
Skipping val split for custom: no images found in /content/detect-1/valid/images
MERGE COMPLETE
Total classes in merged dataset: 85
Merged dataset located at: /content/merged_dataset


## Step 4: Validate the Merged Dataset

To ensure the merging process was successful and the dataset is properly structured, we'll run a quick validation check. This script will count the number of images and label files in the `train` and `val` splits of the `merged_dataset`.

In [6]:
print(f"Validating merged dataset at {MERGED_DIR}...")

total_merged_images = 0
total_merged_labels = 0

for split in ["train", "val"]:
    img_dir = os.path.join(MERGED_DIR, "images", split)
    lbl_dir = os.path.join(MERGED_DIR, "labels", split)

    imgs_count = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    lbls_count = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0

    print(f"\n--- {split.capitalize()} Split ---")
    print(f"Images: {imgs_count}")
    print(f"Labels: {lbls_count}")

    total_merged_images += imgs_count
    total_merged_labels += lbls_count

print("\n--- Total Merged Dataset ---")
print(f"Total Images: {total_merged_images}")
print(f"Total Labels: {total_merged_labels}")

if total_merged_images > 0 and total_merged_images == total_merged_labels:
    print("Validation successful: Images and labels counts match and are non-zero.")
elif total_merged_images == 0:
    print("Validation failed: No images found in the merged dataset.")
else:
    print("Validation warning: Image and label counts do not match. Check for missing label files.")

Validating merged dataset at /content/merged_dataset...

--- Train Split ---
Images: 5010
Labels: 5010

--- Val Split ---
Images: 1200
Labels: 1200

--- Total Merged Dataset ---
Total Images: 6210
Total Labels: 6210
Validation successful: Images and labels counts match and are non-zero.


## Step 5: Train YOLOv8 Model

Finally, with the merged dataset prepared, validated, and its `data.yaml` configured, you can now proceed to train a YOLOv8 model. This cell demonstrates how to load a pre-trained YOLOv8n (nano) model and initiate training using your `merged_dataset`.

In [ ]:
from ultralytics import YOLO

# Load a pre-trained YOLOv8n model
model = YOLO('yolov8n.pt')  # You can choose other models like 'yolov8s.pt', 'yolov8m.pt', etc.

print(f"Starting YOLOv8 training with dataset: {MERGED_DIR}/data.yaml")

# Train the model
results = model.train(
    data=os.path.join(MERGED_DIR, "data.yaml"),
    epochs=5, # You can adjust the number of epochs
    imgsz=640, # Image size
    batch=16,  # Batch size
    name='merged_dataset_yolov8n_training' # Name for the training run
)

print("YOLOv8 training complete!")
print("Results are saved in: ", model.trainer.save_dir)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Starting YOLOv8 training with dataset: /content/merged_dataset/data.yaml
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/merged_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript,